In [1]:
import pandas as pd

In [2]:
def calculate_metrics(rows):
    # Convert list of dicts to DataFrame
    df = pd.DataFrame(rows)
    
    # Pre-calculate totals for each row
    df["Support"] = df["TP"] + df["FN"]
    df["Pred_Count"] = df["TP"] + df["FP"]

    # Calculate individual metrics
    # We use .where() or replacement to handle division by zero safely
    df["Precision"] = df["TP"] / df["Pred_Count"].replace(0, 1)
    df["Recall"] = df["TP"] / df["Support"].replace(0, 1)
    
    denom = (df["Precision"] + df["Recall"]).replace(0, 1)
    df["F1-Score"] = 2 * (df["Precision"] * df["Recall"]) / denom

    # Calculate Weighted Averages based on Support
    total_support = df["Support"].sum()
    
    if total_support > 0:
        w_prec = (df["Precision"] * df["Support"]).sum() / total_support
        w_rec  = (df["Recall"] * df["Support"]).sum() / total_support
        w_f1   = (df["F1-Score"] * df["Support"]).sum() / total_support
    else:
        w_prec = w_rec = w_f1 = 0.0

    # Create the summary row
    avg_row = pd.DataFrame([{
        "Antipattern": "WEIGHTED AVERAGE",
        "TP": df["TP"].sum(),
        "FP": df["FP"].sum(),
        "FN": df["FN"].sum(),
        "Support": total_support,
        "Pred_Count": df["Pred_Count"].sum(),
        "Precision": w_prec,
        "Recall": w_rec,
        "F1-Score": w_f1
    }])
    
    # Combine and return
    return pd.concat([df, avg_row], ignore_index=True).round(4)

In [3]:
data = [
    {"Antipattern": "Implicit Columns", "TP": 83, "FP": 3, "FN": 0},
    {"Antipattern": "ID Required", "TP": 100, "FP": 0, "FN": 5},
    {"Antipattern": "Keyless Entry", "TP": 34, "FP": 10, "FN": 0},
    {"Antipattern": "Fear of the Unknown", "TP": 32, "FP": 11, "FN": 2},
    {"Antipattern": "31 Flavors", "TP": 15, "FP": 0, "FN": 0},
    {"Antipattern": "Poor Man's Search Engine", "TP": 10, "FP": 0, "FN": 1},
    {"Antipattern": "Rounding Errors", "TP": 9, "FP": 0, "FN": 6}
]
calculate_metrics(data)

,Antipattern,TP,FP,FN,Support,Pred_Count,Precision,Recall,F1-Score
0,Implicit Columns,83,3,0,83,86,0.9651,1.0000,0.9822
1,ID Required,100,0,5,105,100,1.0000,0.9524,0.9756
2,Keyless Entry,34,10,0,34,44,0.7727,1.0000,0.8718
3,Fear of the Unknown,32,11,2,34,43,0.7442,0.9412,0.8312
4,31 Flavors,15,0,0,15,15,1.0000,1.0000,1.0000
5,Poor Man's Search Engine,10,0,1,11,10,1.0000,0.9091,0.9524
6,Rounding Errors,9,0,6,15,9,1.0000,0.6000,0.7500
7,WEIGHTED AVERAGE,283,24,14,297,307,0.9349,0.9529,0.9380
